# Notebook 1 : filtrage du dataset Steam Reviews

Objectif : produire un fichier Parquet filtre, sous 10 Go, a partir du CSV brut (16 Go compresses, 21 Go decompresses).

Le fichier filtre est sauvegarde sur Google Drive pour survivre a la fin de la session Colab et etre repris dans le notebook d'analyse. Si ce fichier existe deja sur Drive, l'etape 2bis permet de le charger directement sans repasser par le telechargement et le filtrage.

## 1. Authentification Kaggle

Token stocke dans les secrets Colab, jamais en clair dans le notebook. Necessaire uniquement si le Parquet filtre n'existe pas encore (etape 2bis negative).

1. Generer un token sur kaggle.com (Account, Create New API Token)
2. Dans Colab, panneau Secrets (icone cle a gauche)
3. Creer un secret KAGGLE_API_TOKEN, activer l'acces pour ce notebook

In [ ]:
from google.colab import userdata
import os

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

!pip install -q kaggle

## 2. Google Drive

A executer dans chaque nouvelle session, y compris quand un nouveau notebook est uploade : le disque de la VM Colab est vide a chaque demarrage, seul Drive persiste entre les sessions.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/projet_steam_reviews"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

FILTERED_PARQUET_PATH = os.path.join(DRIVE_OUTPUT_DIR, "steam_reviews_filtered.parquet")

## 2bis. Verification d'un fichier filtre existant

Controle de presence et de taille avant de tenter une lecture. Un fichier dont la taille est anormalement basse (quelques Ko ou Mo, alors que le dernier export valide faisait 160 Mo ou plus) indique une ecriture interrompue par un plantage de session : dans ce cas, le supprimer et repasser par le telechargement plutot que de chercher a le reparer.

In [ ]:
!pip install -q pyarrow
import pandas as pd

df_filtered = None

if os.path.exists(FILTERED_PARQUET_PATH):
    taille_mo = os.path.getsize(FILTERED_PARQUET_PATH) / 1e6
    print("Fichier trouve, taille :", round(taille_mo, 1), "Mo")
    if taille_mo < 1:
        print("Taille suspecte, fichier probablement incomplet. Ne pas charger, repasser par le telechargement.")
    else:
        df_filtered = pd.read_parquet(FILTERED_PARQUET_PATH)
        print("Charge :", df_filtered.shape[0], "lignes,", df_filtered.shape[1], "colonnes")
else:
    print("Aucun fichier filtre trouve sur Drive, passer par le telechargement et le filtrage ci-dessous.")

Si la cellule precedente a charge df_filtered avec succes, le reste du notebook (etapes 3 a 11) n'est plus necessaire : passer directement au notebook d'analyse. Les etapes suivantes ne servent qu'a regenerer le fichier filtre depuis zero.

## 3. Recuperation du CSV brut

Le CSV brut est cherche d'abord sur Drive (copie persistante d'une session precedente). S'il n'y est pas, telechargement depuis Kaggle sur le disque de la VM, puis copie vers Drive pour ne plus jamais avoir a retelecharger apres un plantage de session.

Le disque de la VM reste la source de travail pour le filtrage (lecture plus rapide que depuis Drive), Drive ne sert que de sauvegarde entre les sessions.

In [ ]:
RAW_CSV_DRIVE_PATH = os.path.join(DRIVE_OUTPUT_DIR, "all_reviews_raw.csv")
CSV_PATH = "steam_data/all_reviews/all_reviews.csv"

if os.path.exists(RAW_CSV_DRIVE_PATH):
    print("CSV brut trouve sur Drive, copie vers le disque de la VM.")
    os.makedirs("steam_data/all_reviews", exist_ok=True)
    !cp "$RAW_CSV_DRIVE_PATH" "$CSV_PATH"
else:
    print("Aucune copie sur Drive, telechargement depuis Kaggle.")
    !kaggle datasets download -d kieranpoc/steam-reviews
    !unzip -o steam-reviews.zip -d steam_data
    print("Sauvegarde du CSV brut sur Drive pour eviter un futur retelechargement.")
    !cp "$CSV_PATH" "$RAW_CSV_DRIVE_PATH"

!ls -lhR steam_data

## 4. Colonnes du fichier et controle de taille

Fichier reel : steam_data/all_reviews/all_reviews.csv. La colonne contenant le nom du jeu s'appelle game (pas app_name).

Le fichier complet decompresse doit peser environ 21 Go. Une taille nettement inferieure indique un telechargement tronque, par exemple suite a une coupure reseau pendant le telechargement Kaggle : dans ce cas, supprimer le fichier local et sur Drive, puis relancer l'etape 3 entierement plutot que de continuer sur une base incomplete.

In [ ]:
taille_go = os.path.getsize(CSV_PATH) / 1e9
print("Taille du CSV brut :", round(taille_go, 1), "Go")
if taille_go < 19:
    print("Taille suspecte, attendu environ 21 Go. Verifier un eventuel telechargement tronque avant de continuer.")

sample = pd.read_csv(CSV_PATH, nrows=5)
print(sample.columns.tolist())
sample.head()

## 5. Comptage des avis par jeu

Premiere passe sur le fichier complet, restreinte aux colonnes game et language pour rester rapide. Ce comptage sert deux usages : retrouver l'orthographe exacte des jeux a controverse documentee, et identifier les jeux les plus reviewes pour completer le volume sans deviner de noms.

Tester un nom de jeu directement dans TARGET_GAMES sans verification a echoue precedemment : 8 des 14 noms ne correspondaient a rien dans le fichier.

In [ ]:
import pyarrow as pa
import pyarrow.csv as pv
from collections import Counter

CHUNK_SIZE = 1_000_000

def skip_invalid(row):
    return "skip"

read_options = pv.ReadOptions(block_size=CHUNK_SIZE * 200)
parse_options = pv.ParseOptions(delimiter=",", invalid_row_handler=skip_invalid)
count_convert_options = pv.ConvertOptions(include_columns=["game", "language"])

reader = pv.open_csv(
    CSV_PATH,
    read_options=read_options,
    parse_options=parse_options,
    convert_options=count_convert_options,
)

review_counts = Counter()
total_rows_read = 0

for batch in reader:
    chunk = batch.to_pandas()
    total_rows_read += len(chunk)
    chunk = chunk[chunk["language"] == "english"]
    review_counts.update(chunk["game"].value_counts().to_dict())

counts_series = pd.Series(review_counts).sort_values(ascending=False)

print("Lignes lues au total :", total_rows_read)
if total_rows_read < 50_000_000:
    print("Nombre de lignes plus bas qu'attendu (reference precedente : environ 58 millions). Verifier la taille du CSV a l'etape 4 avant de continuer.")
print("Jeux distincts (avis en anglais) :", len(counts_series))
counts_series.head(20)

## 6. Recherche des jeux a controverse documentee

Recherche par mot cle dans les noms reellement presents. Premier essai avec les jeux initialement vises (helldivers, team fortress, pubg, playerunknown, kerbal space program, assassin's creed unity, sonic mania, firewatch, titan souls, stardew) : aucun des dix n'a de correspondance, ce dataset ne couvre que 13 913 jeux sur les plus de 100 000 existants sur Steam, et ces titres n'en font pas partie.

Liste de mots cles remplacee par des controverses reellement presentes dans ce dataset, identifiees a partir du top des jeux les plus reviewes de l'etape 5 (Cyberpunk 2077, Elden Ring, Apex Legends, Destiny 2, Baldur's Gate 3, etc., qui ont chacun connu un episode de mecontentement documente).

In [ ]:
KEYWORDS = [
    "war thunder",
    "grand theft auto",
    "crusader kings",
    "counter-strike",
    "dota",
    "terraria",
    "cyberpunk 2077",
    "elden ring",
    "apex legends",
    "destiny",
    "baldur's gate 3",
    "fall guys",
    "red dead redemption",
    "new world",
    "halo infinite",
    "hades",
]

for keyword in KEYWORDS:
    found = [name for name in counts_series.index if keyword in name.lower()]
    print(keyword, "->", found)

## 7. Liste finale de jeux

CONTROVERSY_GAMES contient les noms exacts confirmes a l'etape 6 (jeu de base uniquement, DLC et bandes originales exclus). TOP_N_POPULAR fixe combien de jeux les plus reviewes s'ajoutent en complement, pour remplir le volume disponible sous 10 Go.

A 150 jeux populaires, le fichier obtenu ne pesait que 0.98 Go de memoire, tres loin du seuil disponible. TOP_N_POPULAR releve a 1000 pour se rapprocher de 5 a 8 Go ; a ajuster a la hausse ou a la baisse selon le resultat obtenu a l'etape 11, en modifiant uniquement cette valeur puis en relancant les etapes 8 a 11 sans repasser par le telechargement.

In [ ]:
CONTROVERSY_GAMES = [
    "War Thunder",
    "Grand Theft Auto V",
    "Crusader Kings III",
    "Crusader Kings II",
    "Counter-Strike 2",
    "Counter-Strike",
    "Counter-Strike: Source",
    "Dota 2",
    "Terraria",
    "Cyberpunk 2077",
    "ELDEN RING",
    "Apex Legends",
    "Destiny 2",
    "Baldur's Gate 3",
    "Fall Guys",
    "Red Dead Redemption 2",
    "New World",
    "Halo Infinite",
    "Hades",
]

TOP_N_POPULAR = 1000
top_popular_games = counts_series.head(TOP_N_POPULAR).index.tolist()

TARGET_GAMES = sorted(set(CONTROVERSY_GAMES) | set(top_popular_games))

manquants = [g for g in CONTROVERSY_GAMES if g not in counts_series.index]
if manquants:
    print("Attention, noms absents de counts_series, a retirer ou corriger :", manquants)

print("Jeux a controverse confirmes :", len(CONTROVERSY_GAMES))
print("Jeux populaires ajoutes :", len(top_popular_games))
print("Total des jeux cibles :", len(TARGET_GAMES))

## 8. Filtrage complet

Deuxieme passe sur le fichier, cette fois avec toutes les colonnes, restreinte a TARGET_GAMES et a la langue anglaise. Lecture par lots pour ne jamais charger les 21 Go en memoire d'un coup.

In [ ]:
full_convert_options = pv.ConvertOptions(column_types={
    "steam_china_location": pa.string(),
    "review": pa.string(),
})

reader = pv.open_csv(
    CSV_PATH,
    read_options=read_options,
    parse_options=parse_options,
    convert_options=full_convert_options,
)

filtered_parts = []
total_rows_read_pass2 = 0

for batch in reader:
    chunk = batch.to_pandas()
    total_rows_read_pass2 += len(chunk)
    chunk = chunk[chunk["language"] == "english"]
    chunk = chunk[chunk["game"].isin(TARGET_GAMES)]
    if len(chunk) > 0:
        filtered_parts.append(chunk)

df_filtered = pd.concat(filtered_parts, ignore_index=True)

print("Lignes lues au total :", total_rows_read_pass2)
print("Lignes conservees apres filtrage :", df_filtered.shape[0])
print("Memoire occupee :", round(df_filtered.memory_usage(deep=True).sum() / 1e9, 2), "Go")

## 9. Repartition par jeu

Verifier que chaque jeu de CONTROVERSY_GAMES apparait avec un nombre de lignes non nul. Un jeu absent ici alors qu'il a ete trouve a l'etape 6 indique une difference entre les deux passes (nom modifie entre temps, ou erreur de copie depuis la sortie de l'etape 6).

In [ ]:
print(df_filtered["game"].value_counts())

## 10. Export Parquet vers Google Drive

Compression Snappy, stockage en colonnes. Le fichier est ecrit directement sur Drive pour etre repris tel quel par le notebook d'analyse, y compris dans une session future ou un autre notebook (voir etape 2bis).

In [ ]:
df_filtered.to_parquet(FILTERED_PARQUET_PATH, compression="snappy")

!ls -lh "$FILTERED_PARQUET_PATH"

## 11. Verification du seuil

Si le fichier reste loin sous 10 Go, augmenter TOP_N_POPULAR a l'etape 7 et relancer les etapes 8 a 11. Si le fichier depasse 10 Go, reduire TOP_N_POPULAR plutot que d'echantillonner aleatoirement, pour ne pas diluer les jeux a controverse.

In [ ]:
size_go = os.path.getsize(FILTERED_PARQUET_PATH) / 1e9
print("Taille finale du fichier filtre :", round(size_go, 2), "Go")
if size_go > 10:
    print("Au dessus du seuil gratuit BigQuery, reduire TOP_N_POPULAR.")
else:
    print("Sous le seuil, fichier pret pour le notebook d'analyse.")